# M2 - procura temporal (exploração)
Viagens limpas vêm do `m1_clean.py` (igual ao que o `main.py` faz). Notas em `docs/notas-tomas.md`.

In [1]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .master("spark://spark-master:7077")
         .appName("m2-dev")
         .config("spark.driver.memory", "1g")
         .config("spark.executor.memory", "1200m")
         .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/22 16:04:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import sys, importlib, contextlib
from types import SimpleNamespace
from pyspark.sql import functions as F
sys.path.insert(0, "/project/src")
import taxi.m1_clean as m1

class NoProf:                      # profiler que não mede nada (o verdadeiro só no main.py)
    def profile(self, name):
        return contextlib.nullcontext()

a = SimpleNamespace(input="/project/data/raw", zones="/project/data/taxi_zone_lookup.csv",
                    output="/project/output", years=[2019], months="1-1", fraction=1.0)
trips, zones = m1.run(spark, a, set(), NoProf())

## Bloco M2-1: join com as zonas + colunas novas

In [3]:
pu = zones.select(F.col("LocationID").alias("PULocationID"), F.col("Borough").alias("borough"))
t = trips.join(pu, "PULocationID")

dur = (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
t = (t.withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))   # 1 = domingo
      .withColumn("hour_of_day", F.hour("tpep_pickup_datetime"))
      .withColumn("duration_min", dur))

t.select("PULocationID", "borough", "tpep_pickup_datetime", "day_of_week", "hour_of_day", "duration_min").show(5)

+------------+---------+--------------------+-----------+-----------+------------------+
|PULocationID|  borough|tpep_pickup_datetime|day_of_week|hour_of_day|      duration_min|
+------------+---------+--------------------+-----------+-----------+------------------+
|         151|Manhattan| 2019-01-01 00:46:40|          3|          0| 6.666666666666667|
|         239|Manhattan| 2019-01-01 00:59:47|          3|          0|              19.2|
|         163|Manhattan| 2019-01-01 00:21:28|          3|          0|              7.15|
|         229|Manhattan| 2019-01-01 00:32:01|          3|          0|13.633333333333333|
|         141|Manhattan| 2019-01-01 00:57:32|          3|          0|              12.0|
+------------+---------+--------------------+-----------+-----------+------------------+
only showing top 5 rows


In [4]:
t.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [PULocationID#25, tpep_pickup_datetime#23, tpep_dropoff_datetime#24, DOLocationID#26, passenger_count#27, trip_distance#8, RatecodeID#29, payment_type#30, fare_amount#14, tip_amount#17, total_amount#20, 2019 AS year#34, 1 AS month#35, borough#98, dayofweek(cast(tpep_pickup_datetime#23 as date)) AS day_of_week#100, hour(tpep_pickup_datetime#23, Some(Etc/UTC)) AS hour_of_day#101, (cast((unix_timestamp(tpep_dropoff_datetime#24, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), true) - unix_timestamp(tpep_pickup_datetime#23, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), true)) as double) / 60.0) AS duration_min#102]
   +- BroadcastHashJoin [PULocationID#25], [PULocationID#97], Inner, BuildRight, false, false
      :- Project [cast(tpep_pickup_datetime#5 as timestamp) AS tpep_pickup_datetime#23, cast(tpep_dropoff_datetime#6 as timestamp) AS tpep_dropoff_datetime#24, cast(PULocationID#11L as int) AS PULocationID#25, cast(DOLocationID#12L as int) AS 

In [5]:
res = (t.groupBy("borough", "day_of_week", "hour_of_day")
        .agg(F.count("*").alias("trip_count"),
             F.avg("fare_amount").alias("mean_fare"),
             F.avg("duration_min").alias("mean_duration_min"),
             F.avg("trip_distance").alias("mean_distance")))

res.orderBy(F.desc("trip_count")).show(5)

[Stage 10:>                                                         (0 + 1) / 1]

+---------+-----------+-----------+----------+------------------+------------------+------------------+
|  borough|day_of_week|hour_of_day|trip_count|         mean_fare| mean_duration_min|     mean_distance|
+---------+-----------+-----------+----------+------------------+------------------+------------------+
|Manhattan|          5|         18|     82894|10.269646898448622|15.762448428112918|1.9570541896880356|
|Manhattan|          4|         18|     78879|10.162169525475727|15.353525441921452| 1.978021526642067|
|Manhattan|          5|         19|     77309|10.136695598183909|14.240052688992483| 2.062180858632253|
|Manhattan|          3|         18|     73181|10.055317910386574| 14.50016169953039|1.9701049452726669|
|Manhattan|          4|         19|     73029| 9.885265305563543|13.400595653781222| 2.069617001465172|
+---------+-----------+-----------+----------+------------------+------------------+------------------+
only showing top 5 rows


In [6]:
res.write.mode("overwrite").partitionBy("borough").parquet("/project/output/m2")